# 🌳 CodTech Internship — Task 1
# Decision Tree Implementation

---

**Name:** Omkar Mohanty 
**Internship:** CodTech IT Solutions  
**Task:** Build and Visualize a Decision Tree Model using Scikit-learn  
**Dataset:** Breast Cancer Wisconsin Dataset (built-in sklearn)  
**Objective:** Classify tumors as Malignant or Benign

---

## 📦 Section 1 — Import Libraries

In [ ]:
# Core Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn — Dataset
from sklearn.datasets import load_breast_cancer

# Sklearn — Preprocessing & Splitting
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler

# Sklearn — Model
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

# Sklearn — Evaluation
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✅ All libraries imported successfully!')

---
## 📊 Section 2 — Load & Explore Dataset

In [ ]:
# Load Dataset
data = load_breast_cancer()

# Convert to DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print('📌 Dataset: Breast Cancer Wisconsin')
print(f'📐 Shape   : {df.shape}')
print(f'🔢 Features: {len(data.feature_names)}')
print(f'🎯 Classes : {list(data.target_names)}')
print()
df.head()

In [ ]:
# Basic Info
print('=== Dataset Info ===')
print(df.info())

In [ ]:
# Statistical Summary
print('=== Statistical Summary ===')
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# Check for Missing Values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else '✅ No missing values found!')

In [ ]:
# Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count Plot
counts = df['diagnosis'].value_counts()
bars = axes[0].bar(counts.index, counts.values,
                   color=['#E74C3C', '#2ECC71'], edgecolor='black', linewidth=1.2)
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Diagnosis')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(val), ha='center', fontweight='bold', fontsize=12)

# Pie Chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#E74C3C', '#2ECC71'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')

plt.suptitle('Target Class Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Correlation Heatmap (Top 10 Features)
top_features = df[data.feature_names[:10]].copy()
top_features['target'] = df['target']

plt.figure(figsize=(12, 9))
corr = top_features.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Heatmap (Top 10 Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Distribution by Class (Top 6 Features)
top6 = list(data.feature_names[:6])
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(top6):
    for label, color in zip(['Malignant', 'Benign'], ['#E74C3C', '#2ECC71']):
        subset = df[df['diagnosis'] == label][feature]
        axes[i].hist(subset, bins=25, alpha=0.6, label=label, color=color, edgecolor='black', linewidth=0.5)
    axes[i].set_title(feature, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

plt.suptitle('Feature Distribution by Class (Top 6 Features)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ⚙️ Section 3 — Data Preprocessing & Train/Test Split

In [ ]:
# Define Features and Target
X = data.data
y = data.target
feature_names = list(data.feature_names)
class_names   = list(data.target_names)

# Train / Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Split Complete')
print(f'   Training samples : {X_train.shape[0]}')
print(f'   Testing samples  : {X_test.shape[0]}')
print(f'   Features         : {X_train.shape[1]}')
print()
print(f'   Train class distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'   Test  class distribution: {dict(zip(*np.unique(y_test,  return_counts=True)))}')

---
## 🌳 Section 4 — Build the Decision Tree Model

In [ ]:
# Build Decision Tree Classifier
dt_model = DecisionTreeClassifier(
    criterion='entropy',      # Information Gain splitting
    max_depth=4,              # Limit depth to avoid overfitting
    min_samples_split=10,     # Min samples required to split a node
    min_samples_leaf=5,       # Min samples required at each leaf
    random_state=42
)

# Train the Model
dt_model.fit(X_train, y_train)

print('✅ Decision Tree Model Trained Successfully!')
print(f'   Tree Depth   : {dt_model.get_depth()}')
print(f'   No. of Leaves: {dt_model.get_n_leaves()}')

---
## 📈 Section 5 — Model Evaluation

In [ ]:
# Predictions
y_pred       = dt_model.predict(X_test)
y_pred_prob  = dt_model.predict_proba(X_test)[:, 1]

# Accuracy
train_acc = accuracy_score(y_train, dt_model.predict(X_train))
test_acc  = accuracy_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_pred_prob)

print('=' * 45)
print('         MODEL PERFORMANCE SUMMARY')
print('=' * 45)
print(f'  Training Accuracy : {train_acc * 100:.2f}%')
print(f'  Testing  Accuracy : {test_acc  * 100:.2f}%')
print(f'  ROC-AUC Score     : {roc_auc:.4f}')
print('=' * 45)
print()
print('--- Classification Report ---')
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')

# Normalized Confusion Matrix
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)
disp2.plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#E74C3C', lw=2.5,
         label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.1, color='#E74C3C')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC-AUC Curve — Decision Tree', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Cross Validation (10-Fold)
cv_scores = cross_val_score(dt_model, X, y, cv=10, scoring='accuracy')

plt.figure(figsize=(10, 5))
bars = plt.bar(range(1, 11), cv_scores * 100, color='steelblue',
               edgecolor='black', linewidth=1)
plt.axhline(y=cv_scores.mean() * 100, color='red', linestyle='--',
            linewidth=2, label=f'Mean = {cv_scores.mean()*100:.2f}%')
plt.fill_between(range(1, 11),
                 (cv_scores.mean() - cv_scores.std()) * 100,
                 (cv_scores.mean() + cv_scores.std()) * 100,
                 alpha=0.15, color='red', label=f'±1 Std = {cv_scores.std()*100:.2f}%')
for bar, score in zip(bars, cv_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{score*100:.1f}%', ha='center', fontsize=9, fontweight='bold')
plt.xlabel('Fold Number', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('10-Fold Cross Validation Accuracy', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.ylim(80, 105)
plt.tight_layout()
plt.show()

print(f'\n📊 Cross-Validation Results:')
print(f'   Mean Accuracy : {cv_scores.mean()*100:.2f}%')
print(f'   Std Deviation : ±{cv_scores.std()*100:.2f}%')
print(f'   Min Accuracy  : {cv_scores.min()*100:.2f}%')
print(f'   Max Accuracy  : {cv_scores.max()*100:.2f}%')

---
## 🌲 Section 6 — Decision Tree Visualization

In [ ]:
# Full Decision Tree Visualization
plt.figure(figsize=(28, 14))
plot_tree(
    dt_model,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=9,
    impurity=True,
    proportion=False
)
plt.title('🌳 Decision Tree — Breast Cancer Classification\n(Criterion: Entropy | Max Depth: 4)',
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('decision_tree_full.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Tree saved as decision_tree_full.png')

In [ ]:
# Simplified Shallow Tree (Depth 2) — Easy to Read
dt_shallow = DecisionTreeClassifier(
    criterion='entropy', max_depth=2, random_state=42
)
dt_shallow.fit(X_train, y_train)

plt.figure(figsize=(16, 8))
plot_tree(
    dt_shallow,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=11,
    impurity=True
)
plt.title('🌱 Simplified Decision Tree (Max Depth = 2)',
          fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Text Representation of the Tree
print('=== DECISION TREE TEXT RULES ===')
tree_rules = export_text(dt_model, feature_names=feature_names)
print(tree_rules)

---
## 📌 Section 7 — Feature Importance Analysis

In [ ]:
# Feature Importances
importances = pd.Series(
    dt_model.feature_importances_,
    index=feature_names
).sort_values(ascending=True)

# Filter non-zero importances
importances_nonzero = importances[importances > 0]

plt.figure(figsize=(10, 7))
colors = ['#E74C3C' if v == importances_nonzero.max() else 'steelblue'
          for v in importances_nonzero.values]
bars = plt.barh(importances_nonzero.index, importances_nonzero.values,
                color=colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, importances_nonzero.values):
    plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9, fontweight='bold')
plt.xlabel('Feature Importance (Gini/Entropy Weight)', fontsize=12)
plt.title('Feature Importances — Decision Tree\n(Red = Most Important Feature)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🔝 Top 5 Most Important Features:')
top5 = importances.sort_values(ascending=False).head(5)
for i, (feat, imp) in enumerate(top5.items(), 1):
    print(f'   {i}. {feat}: {imp:.4f} ({imp*100:.2f}%)')

---
## 🔍 Section 8 — Hyperparameter Tuning (GridSearchCV)

In [ ]:
# Define Hyperparameter Grid
param_grid = {
    'criterion'        : ['gini', 'entropy'],
    'max_depth'        : [3, 4, 5, 6, 7, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf' : [1, 3, 5, 7]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=10,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train, y_train)

print('✅ GridSearchCV Complete!')
print(f'\n🏆 Best Parameters  : {grid_search.best_params_}')
print(f'📊 Best CV Accuracy : {grid_search.best_score_*100:.2f}%')

In [ ]:
# Optimized Model using Best Parameters
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

best_train_acc = accuracy_score(y_train, best_model.predict(X_train))
best_test_acc  = accuracy_score(y_test, y_pred_best)

print('=' * 50)
print('      OPTIMIZED MODEL — PERFORMANCE COMPARISON')
print('=' * 50)
print(f'  Baseline  Test Accuracy : {test_acc  * 100:.2f}%')
print(f'  Optimized Test Accuracy : {best_test_acc * 100:.2f}%')
improvement = (best_test_acc - test_acc) * 100
print(f'  Improvement             : {improvement:+.2f}%')
print('=' * 50)
print()
print('--- Optimized Classification Report ---')
print(classification_report(y_test, y_pred_best, target_names=class_names))

In [ ]:
# Model Comparison — Baseline vs Optimized
models     = ['Baseline\nDecision Tree', 'Tuned\nDecision Tree']
train_accs = [train_acc * 100, best_train_acc * 100]
test_accs  = [test_acc  * 100, best_test_acc  * 100]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))
bars1 = ax.bar(x - width/2, train_accs, width, label='Train Accuracy',
               color='#3498DB', edgecolor='black')
bars2 = ax.bar(x + width/2, test_accs,  width, label='Test Accuracy',
               color='#2ECC71', edgecolor='black')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.2f}%', ha='center', fontsize=10, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison: Baseline vs Tuned Decision Tree',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=12)
ax.set_ylim(85, 103)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 🪓 Section 9 — Cost-Complexity Pruning

In [ ]:
# Find Optimal Alpha for Pruning
pruning_model = DecisionTreeClassifier(random_state=42)
path = pruning_model.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[::3]  # sample every 3rd alpha for speed

train_scores, test_scores = [], []

for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, clf.predict(X_train)))
    test_scores.append(accuracy_score(y_test, clf.predict(X_test)))

# Plot
plt.figure(figsize=(10, 6))
plt.plot(ccp_alphas, train_scores, 'o-', label='Train Accuracy',
         color='#3498DB', lw=2, markersize=5)
plt.plot(ccp_alphas, test_scores,  's-', label='Test Accuracy',
         color='#E74C3C',  lw=2, markersize=5)

# Mark best test accuracy
best_idx   = np.argmax(test_scores)
best_alpha = ccp_alphas[best_idx]
plt.axvline(x=best_alpha, color='green', linestyle='--', lw=1.5,
            label=f'Best Alpha = {best_alpha:.4f}')

plt.xlabel('CCP Alpha (Pruning Strength)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Effect of Pruning (CCP Alpha) on Model Accuracy',
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'\n✂️  Optimal CCP Alpha  : {best_alpha:.4f}')
print(f'   Best Test Accuracy : {max(test_scores)*100:.2f}%')

---
## 📝 Section 10 — Summary & Conclusion

In [ ]:
# Final Summary Table
summary = {
    'Metric': [
        'Dataset',
        'Total Samples',
        'Features Used',
        'Train/Test Split',
        'Model',
        'Criterion',
        'Max Depth',
        'Baseline Test Accuracy',
        'Tuned Test Accuracy',
        'CV Mean Accuracy (10-fold)',
        'ROC-AUC Score',
        'Most Important Feature'
    ],
    'Value': [
        'Breast Cancer Wisconsin',
        str(len(X)),
        str(X.shape[1]),
        '80% / 20%',
        'Decision Tree Classifier',
        'Entropy',
        str(dt_model.get_depth()),
        f'{test_acc*100:.2f}%',
        f'{best_test_acc*100:.2f}%',
        f'{cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%',
        f'{roc_auc:.4f}',
        importances.sort_values(ascending=False).index[0]
    ]
}

summary_df = pd.DataFrame(summary)
print('=' * 60)
print('           FINAL PROJECT SUMMARY')
print('=' * 60)
print(summary_df.to_string(index=False))
print('=' * 60)

---

## ✅ Conclusion

In this project, we successfully implemented a **Decision Tree Classifier** using **Scikit-learn** on the **Breast Cancer Wisconsin Dataset** to classify tumors as Malignant or Benign.

### Key Findings:

1. **Model Performance**: The Decision Tree achieved strong classification accuracy on the test set, demonstrating its effectiveness for this medical classification problem.

2. **Important Features**: The most influential features for prediction were related to **worst radius**, **worst concave points**, and **mean concave points** — all characteristics of tumor cell nuclei.

3. **Hyperparameter Tuning**: GridSearchCV improved model generalization by finding the optimal combination of criterion, max depth, and minimum samples parameters.

4. **Cross-Validation**: 10-fold cross-validation confirmed the model's consistency and robustness across different data subsets.

5. **Pruning**: Cost-complexity pruning analysis helped identify the ideal tree simplicity vs accuracy tradeoff.

### Limitations:
- Decision trees can still overfit on noisy datasets — ensemble methods like **Random Forest** or **Gradient Boosting** would further improve performance.
- Feature scaling was not required for Decision Trees, but would be needed for other algorithms.

---
*CodTech Internship | Task 1 | Decision Tree Implementation*